# Exercise 10 : Evaluating a System of Agents

- A final-answer check scores facts in the reply ; it can miss a false claim or reject a paraphrase
- Trajectory match compares tool calls ; strict and unordered catch different failures
- A specialist that passes does not guarantee the desk keeps every fact

You will score a small Aurelia multi-agent desk on six labelled cases : final
answers, trajectory match, and one score per agent. All data is made up.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [ ]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.5", "langchain-core==1.6.10", "langchain-openai==1.7.1", "langgraph==1.2.15", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.24", "langchain-quickjs==0.3.9", "langsmith==0.14.8", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    THINKING_OFF = {"reasoning": {"effort": "none"}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# New OpenRouter accounts may send 20 requests a minute per model.
# One shared limiter keeps parallel agents under it ; raise COURSE_RPS when the cap lifts
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "0.3")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    kw.setdefault("temperature", 0)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

## Step 1 : The desk and the cases

In [ ]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage
from langchain.tools import tool
from rank_bm25 import BM25Okapi

DB = Path(data("capstone/service.db")).resolve()
MANUALS = Path(data_folder("capstone/manuals"))
PAGES = [(p.name, p.read_text(encoding="utf-8")) for p in sorted(Path(MANUALS).glob("*.md"))]
BM25 = BM25Okapi([text.lower().split() for _, text in PAGES])
LIMIT = {"recursion_limit": 24}


@tool
def run_sql(query: str) -> str:
    """Run one read-only SELECT on the Aurelia service database."""
    if not query.lstrip().lower().startswith("select"):
        return "Only SELECT statements are allowed."
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        try:
            return json.dumps(db.execute(query).fetchmany(30))
        except sqlite3.Error as exc:
            return f"SQL error : {exc}"


@tool
def search_manuals(query: str) -> str:
    """Keyword search over Aurelia manuals and bulletins. Returns the three best documents."""
    scores = BM25.get_scores(query.lower().split())
    best = sorted(range(len(PAGES)), key=lambda i: -scores[i])[:3]
    return "\n\n".join(f"[{PAGES[i][0]}]\n{PAGES[i][1][:900]}" for i in best)


SYSTEM = ("You support Aurelia field engineers. Use tools for facts. Be brief. "
          "Preserve part numbers, stock counts and depot names.")
parts_agent = create_agent(llm, tools=[run_sql], system_prompt=SYSTEM, name="parts")
manuals_agent = create_agent(llm, tools=[search_manuals], system_prompt=SYSTEM, name="manuals")
desk = create_agent(llm, tools=[run_sql, search_manuals], system_prompt=SYSTEM, name="desk")

CASES = [json.loads(line) for line in
         Path(data("capstone/eval.jsonl")).read_text(encoding="utf-8").splitlines()]
SELECTED_IDS = ["db-01", "db-03", "man-01", "man-07", "mix-01", "mix-02"]
SELECTED = [c for c in CASES if c["id"] in SELECTED_IDS]
print(len(SELECTED), "selected cases :", [c["id"] for c in SELECTED])

In [ ]:
print("check :", "PASS" if len(SELECTED) == 6 else "FAIL")
assert len(SELECTED) == 6

## Step 2 : Final-answer scores

Complete `score_answer` so it checks expected facts and expected tools on a
desk result.

In [ ]:
def run_desk(case):
    return desk.invoke({"messages": [{"role": "user", "content": case["question"]}]}, LIMIT)


def score_answer(case, result):
    answer = str(result["messages"][-1].content)
    used = [m.name for m in result["messages"] if m.type == ____]
    facts_ok = any(fact.lower() in answer.____() for fact in case["answer_contains"])
    tools_ok = set(case["expected_tools"]) <= set(____)
    return {"id": case["id"], "facts_ok": facts_ok, "tools_ok": tools_ok,
            "used": used, "answer": answer}


desk_results = []
for case in SELECTED:
    result = run_desk(case)
    row = score_answer(case, result)
    row["messages"] = result["messages"]
    desk_results.append(row)
    print(f"{row['id']:8} facts {row['facts_ok']!s:5} tools {row['tools_ok']!s:5} {row['used']}")
passed = sum(r["facts_ok"] and r["tools_ok"] for r in desk_results)
print(f"{passed}/{len(desk_results)} passed")

<details><summary>Hint</summary>

`"tool"`, `lower()` and `used`.

</details>

In [ ]:
print("check :", "PASS" if passed >= 4 else "FAIL", f"{passed}/6")
assert passed >= 3

## Step 3 : Trajectory match

Compare the live trajectory for the first case with a reference. Use
`tool_args_match_mode="ignore"` so the lesson is strict against unordered,
not empty reference args. Then score the two fixed fixtures that separate
call order from call coverage.

In [ ]:
from agentevals.trajectory.match import create_trajectory_match_evaluator

# man-01 is a single-tool case ; db-01 often over-calls and never matches unordered.
live = next(r for r in desk_results if r["id"] == "man-01")
reference_calls = [
    {"name": name, "args": {}, "id": f"ref-{i}"}
    for i, name in enumerate(next(c["expected_tools"] for c in SELECTED if c["id"] == "man-01"))
]
reference = [HumanMessage(next(c["question"] for c in SELECTED if c["id"] == "man-01")),
             AIMessage("", tool_calls=reference_calls)]

print("live scores (man-01)")
live_scores = {}
for mode in ("strict", "unordered"):
    evaluator = create_trajectory_match_evaluator(
        trajectory_match_mode=mode, tool_args_match_mode="ignore")
    live_scores[mode] = evaluator(outputs=live["messages"], reference_outputs=reference)["score"]
    print(mode, ":", live_scores[mode])

call_a = {"name": "run_sql", "args": {"query": "SELECT 1"}, "id": "a"}
call_b = {"name": "search_manuals", "args": {"query": "fan"}, "id": "b"}
both = [AIMessage("", tool_calls=[call_a, call_b])]
reversed_calls = [AIMessage("", tool_calls=[call_b, call_a])]
one = [AIMessage("", tool_calls=[call_a])]

print("fixtures")
fixture_scores = {}
for mode in ("strict", "unordered"):
    evaluator = create_trajectory_match_evaluator(
        trajectory_match_mode=____, tool_args_match_mode="ignore")
    fixture_scores[mode] = {
        "reordered": evaluator(outputs=reversed_calls, reference_outputs=both)["score"],
        "one_call": evaluator(outputs=one, reference_outputs=both)["score"],
    }
    print(mode, fixture_scores[mode])

<details><summary>Hint</summary>

`trajectory_match_mode=mode` ; keep `tool_args_match_mode="ignore"`.

</details>

In [ ]:
# Installed strict match permits reordering inside one assistant message.
assert live_scores["unordered"] is True
assert fixture_scores["strict"]["reordered"] is True
assert fixture_scores["unordered"]["reordered"] is True
assert fixture_scores["strict"]["one_call"] is False
assert fixture_scores["unordered"]["one_call"] is False
print("check : PASS live unordered", live_scores, "fixtures", fixture_scores)

## Step 4 : One score per agent

Score the parts agent on a database case, the manuals agent on a manuals case,
and both plus the desk on a mixed case.

In [ ]:
def agent_score(agent, case):
    result = agent.invoke({"messages": [{"role": "user", "content": case["question"]}]}, LIMIT)
    row = score_answer(case, result)
    row["agent"] = agent.name
    return row


db_case = next(c for c in SELECTED if c["id"] == "db-01")
man_case = next(c for c in SELECTED if c["id"] == "man-01")
mix_case = next(c for c in SELECTED if c["id"] == "mix-01")

per_agent = [
    agent_score(____, db_case),
    agent_score(____, man_case),
    agent_score(parts_agent, mix_case),
    agent_score(manuals_agent, mix_case),
]
desk_mix = next(r for r in desk_results if r["id"] == "mix-01")
per_agent.append({"agent": "desk", **{k: desk_mix[k] for k in ("id", "facts_ok", "tools_ok", "used", "answer")}})
for row in per_agent:
    print(f"{row['agent']:8} {row['id']:8} facts {row['facts_ok']!s:5} tools {row['tools_ok']!s:5}")

<details><summary>Hint</summary>

`parts_agent` for the database case and `manuals_agent` for the manuals case.

</details>

In [ ]:
assert any(r["agent"] == "parts" for r in per_agent)
assert any(r["agent"] == "manuals" for r in per_agent)
assert any(r["agent"] == "desk" for r in per_agent)
print("check : PASS per-agent rows")

## If time permits

- Re-score the live man-01 trajectory without `tool_args_match_mode="ignore"` ;
  empty reference args never match a live model's tool call, so the score fails
  for the wrong reason
- Add `subset` and `superset` to the trajectory table
- Replace the fact check with a trajectory judge on one mixed case